In [1]:
# ======================================================
# 📦 Core Imports
# ======================================================
import os, re, h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, filtfilt, resample, find_peaks
import neurokit2 as nk
from scipy.stats import wilcoxon

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print("✅ Imports loaded")


✅ Imports loaded


In [5]:
# --- Respiration file paths (.h5) --- # Baseline recordings
resp_paths_bl = {
    "BL_1_1_d1_2": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_1_d1_2_20250623_103713_merged.h5",
    "BL_1_2_sub1_1": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s1_2_sub_1_1_20250623_120135_merged.h5",
    "BL_2_3_d2_4": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_3_d2_4_20250623_145448_merged.h5",
    "BL_2_4_sub2_3": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s2_4_sub2_3_20250623_141419_merged.h5",
    "BL_3_5_d3_6": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_5_d3_6_20250623_154154_merged.h5",
    "BL_3_6_sub3_5": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s3_6_sub3_5_20250623_172635_merged.h5",
    "BL_4_7_d4_8": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_7_d4_8_20250623_185042_merged.h5",
    "BL_4_8_sub4_7": r"E:\Aim1\AIM1\Cagemate_bl_no_ISO\h5_outputs\BL_s4_8_sub4_7_20250623_180810_merged.h5"
}
resp_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_1_d1_2_20250623_111352_merged.h5",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s1_2_sub1_1_20250623_133932_merged.h5",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_3_d2_4_20250623_151153_merged.h5",
    "CM_2_4_sub2_2": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s2_4_sub2_3_20250623_143348_merged.h5",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_5_d3_6_20250623_170708_merged.h5",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s3_6_sub3_5_20250623_174348_merged.h5",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_7_d4_8_20250623_193718_merged.h5",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\resp_CM_h5\baseline_cagemate_interactions_h5_outputs\CM_s4_8_sub4_7_20250623_182649_merged.h5",
}
# --- BORIS annotation file paths (.csv) --- # Baseline recordings
boris_paths_cm = {
    "CM_1_1_d1_2": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_1_d1_2_20250623_111352.1.csv",
    "CM_1_2_sub1_1": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s1_2_sub1_1_20250623_133932.1_VT.csv",
    "CM_2_3_d2_4": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_3_d2_4_20250623_151153.1.csv",
    "CM_2_4_sub2_3": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s2_4_sub2_3_20250623_143348.1_VT.csv",
    "CM_3_5_d3_6": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_5_d3_6_20250623_160001.csv",
    "CM_3_6_sub3_5": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s3_6_sub3_5_20250623_174348.csv",
    "CM_4_7_d4_8": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_7_d4_8_20250623_193718.csv",
    "CM_4_8_sub4_7": r"E:\Aim1\AIM1\Day1_new\cm_boris\baseline csv\CM_s4_8_sub4_7_20250623_182649.1_VT.csv",
}

print(f"Resp files: {len(resp_paths_cm)} trials")
print(f"Resp files: {len(resp_paths_bl)} trials")
print(f"BORIS files: {len(boris_paths_cm)} trials")

Resp files: 8 trials
Resp files: 8 trials
BORIS files: 8 trials


In [6]:
# --- Respiration file paths (.h5) ---
resp_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_3_p5_3_nRB3_20250622_104059_merged (1).h5",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_3_p5_3_nRB3_20250622_110216_merged.h5",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_8_p5_1_nRB3_20250621_165214_merged.h5",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_8_p5_1_nRB3_20250621_171318_merged.h5",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_1_p5_2_nRB6_20250622_143958_merged.h5",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_1_p5_2_nRB6_20250622_150457_merged.h5",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s1_2_p5_1_nRB6_20250622_170742_merged.h5",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s1_2_p5_1_nRB6_20250622_173049_merged.h5",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s2_4_p5_4_nRB3_20250622_123424_merged.h5",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s2_4_p5_4_nRB3_20250622_125648_merged.h5",
    "RI1_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_5_p5_4_nRB3_20250621_105014_merged.h5",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_5_p5_4_nRB3_20250621_112618_merged.h5",
    # Baseline recordings
    "BLRI_1_1": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_1_p5_2_nRB6_20250622_141846_merged.h5",
    "BLRI_1_2": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s1_2_p5_1_nRB6_20250622_164833_merged.h5",
    "BLRI_2_3": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_3_p5_3_nRB3_20250622_101813_merged.h5",
    "BLRI_2_4": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s2_4_p5_4_nRB3_20250622_121708_merged.h5",
    "BLRI_3_6": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5",
    "BLRI_4_7": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5",
    "BLRI_4_8": r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_8_p5_1_nRB3_20250621_163506_merged.h5",
}

# --- BORIS annotation file paths (.csv) ---
boris_paths = {
    "RI1_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv",
    "RI2_3_6": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv",
    "RI1_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI2_4_7": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv",
    "RI1_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_3_p5_3_nRB3_20250622_104059.1.csv",
    "RI2_2_3": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_3_p5_3_nRB3_20250622_1102116.1.csv",
    "RI1_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_8_p5_1_nRB3_HEEPS.csv",
    "RI2_4_8": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_8_p5_1_nRB3_20250621_HEEPS.csv",
    "RI1_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_1_p5_2_nRB6_20250622_143958.1.csv",
    "RI2_1_1": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_1_p5_2_nRB6_20250622_150457.1.csv",
    "RI1_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s1_2_p5_1_nRB6_20250622_170742.1.csv",
    "RI2_1_2": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s1_2_p5_1_nRB6_20250622_173049.1.csv",
    "RI1_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI1_s2_4_p5_4_nRB3_20250622_123424.1.csv",
    "RI2_2_4": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s2_4_p5_4_nRB3_20250622_125648.1.csv",
    "RI2_3_5": r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_5_p5_4_nRB3_20250621.csv",
}

print(f"Resp files: {len(resp_paths)} trials")
print(f"BORIS files: {len(boris_paths)} trials")

Resp files: 23 trials
BORIS files: 15 trials


In [11]:
# ======================================================
# 🧠 Load & Preprocess Functions
# ======================================================

def load_respiration(h5_path, target_fs=100):
    """Load respiration signal from .h5, filter, and resample."""
    with h5py.File(h5_path, "r") as f:
        key = [k for k in f.keys() if "resp" in k.lower() or "pressure" in k.lower()][0]
        signal = np.array(f[key])
        fs = int(f.attrs.get("sampling_rate", 20000))
    b, a = butter(2, [1/(fs/2), 12/(fs/2)], btype="band")
    filtered = filtfilt(b, a, signal)
    resampled = resample(filtered, int(len(filtered) * target_fs / fs))
    return resampled, target_fs


def extract_resp_features(resp_signal, fs):
    """Extract respiration features for each trial."""
    peaks, _ = find_peaks(resp_signal, distance=fs/20)
    ibi = np.diff(peaks) / fs
    if len(ibi) < 3:
        return pd.Series({"MeanRate_Hz": np.nan, "IBI_std": np.nan,
                          "Amp_mean": np.nan, "Amp_cv": np.nan})
    amp = resp_signal[peaks]
    return pd.Series({
        "MeanRate_Hz": 1 / np.mean(ibi),
        "IBI_std": np.std(ibi),
        "Amp_mean": np.mean(amp),
        "Amp_cv": np.std(amp) / np.mean(amp),
    })


def load_boris(boris_csv):
    """Load BORIS CSV and summarize social behaviors."""
    df = pd.read_csv(boris_csv)
    df.columns = df.columns.str.strip()
    start_col, end_col = ("Start (s)", "Stop (s)") if "Start (s)" in df.columns else ("Start (ms)", "Stop (ms)")
    if start_col.endswith("(ms)"):
        df[start_col] /= 1000; df[end_col] /= 1000
    social_mask = df["Behavior"].str.lower().isin(["facial sniffing", "body sniffing", "anogenital sniffing"])
    df_social = df[social_mask].copy()
    beh_summary = df_social.groupby("Behavior")[[start_col, end_col]].apply(
        lambda x: (x[end_col] - x[start_col]).sum()
    ).reset_index(name="Duration_s")
    total_social = beh_summary["Duration_s"].sum()
    return pd.Series({
        "TotalSocial_s": total_social,
        "Facial_s": beh_summary.query("Behavior=='facial sniffing'")["Duration_s"].sum(),
        "Body_s": beh_summary.query("Behavior=='body sniffing'")["Duration_s"].sum(),
        "AG_s": beh_summary.query("Behavior=='anogenital sniffing'")["Duration_s"].sum(),
    })



In [12]:
rank_fix = {
    "1_1": "Subordinate", "1_2": "Dominant",
    "2_3": "Subordinate", "2_4": "Dominant",
    "3_5": "Subordinate", "3_6": "Dominant",
    "4_7": "Subordinate", "4_8": "Dominant"
}


In [14]:
# ======================================================
# 📂 Load All BORIS CSVs (RI1/RI2 + Cagemate)
# ======================================================

boris_data = {}

# Combine all BORIS dictionaries into one iterable
all_boris_dicts = {
    **boris_paths,     # RI1, RI2, BLRI
    **boris_paths_cm,  # Cagemate
}

print(f"Found {len(all_boris_dicts)} total BORIS files to load.\n")

for label, path in all_boris_dicts.items():
    try:
        df = pd.read_csv(path)
        boris_data[label] = df.copy()
        print(f"✅ Loaded {label:15s} | {df.shape[0]:4d} rows")
    except Exception as e:
        print(f"❌ Error loading {label}: {e}")

print(f"\n📊 Successfully loaded {len(boris_data)} BORIS files total.")


TypeError: 'set' object is not a mapping